In [1]:
import pandas as pd
from datetime import datetime,date
from glob import glob
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib


# 各社有車の稼働時間のまとめ

### 目的
## 社員が予約した内容から社有車の稼働時間を算出するための元データとしてまとめる
## 一つのデータフレームにまとめ、重複したデータは削除する。
## 生のデータフレームを返す(return)

csv_file_path = '../data/'
csv_file_pattern = csv_file_path + 'ご予約リスト_*.csv'
today = date.today().strftime('%Y%m%d')
export_file = f'月別_社有車別_総稼働時間_{today}.xlsx'

def load_and_combine_reservations():
    files = glob(csv_file_pattern)

    row_data = []

    for file in files:
        df = pd.read_csv(file,encoding = 'shift_jis')
        row_data.append(df)

    df = pd.concat(row_data,ignore_index=True)
    df = df.drop_duplicates()

    name_mapping = {
        'キムラヒロカズ':'木村博和',
        '古川':'古川洋子',
        '古川　修理の為':'古川洋子',
        '古川　点検のため':'古川洋子',
        '古谷篤史':'古市篤史',
        '古市　予約':'古市篤史',
        '古市　篤史':'古市篤史',
        '大橋':'大橋邦弘',
        '平賀　博之':'平賀博之',
        '日吉田':'日吉田拓哉',
        '日吉田 拓哉':'日吉田拓哉',
        '春田　真一':'春田真一',
        '木股':'木股寛',
        '木股　寛':'木股寛',
        '柏原':'柏原颯',
        '柏原 颯':'柏原颯',
        '神谷　信文':'神谷信文',
        '落合　則夫':'落合則夫',
        '西口':'西口映美',
        '西口　映美':'西口映美',
        '西村':'西村勝三',
        '西村　勝三':'西村勝三',
        '賚　純一':'賚純一',
        '越野　和久':'越野和久',
        '大川内　幸助':'大川内幸助'
    }
    df['名前_統一'] = df['名前'].map(name_mapping).fillna(df['名前'])
    df[['予約日','稼働時間']] = df['予約日時'].str.split(' ',n=1,expand=True)
    df[['利用開始時刻','利用終了時刻']] = df['稼働時間'].str.split('\r\n~',expand=True)
    df['利用開始時刻'] = df['利用開始時刻'].str.replace('：',':')
    df['利用終了時刻'] = df['利用終了時刻'].str.replace('：',':')
    df['予約日'] = pd.to_datetime(df['予約日'],errors='coerce')
    df['年度'] = df['予約日'].dt.year
    df['月度'] = df['予約日'].dt.month
    df['利用開始日時_str'] = df['予約日'].dt.strftime('%Y-%m-%d') +' '+df['利用開始時刻']
    df['利用終了日時_str'] = df['予約日'].dt.strftime('%Y-%m-%d') +' '+df['利用終了時刻']
    df['利用開始日時'] = pd.to_datetime(df['利用開始日時_str'],errors='coerce')
    df['利用終了日時'] = pd.to_datetime(df['利用終了日時_str'],errors='coerce')

    df = df.drop(columns=['予約日時','利用開始日時_str','利用終了日時_str','予約日','Unnamed: 11'], errors='ignore')

    df['稼働時間'] = (df['利用終了日時'] - df['利用開始日時'])
    df['稼働時間_hour'] = df['稼働時間'].dt.total_seconds()/3600

    three_month_ago = pd.Timestamp.now() - pd.DateOffset(months=3)
    df_recent = df[df['利用終開始日時'] >= three_month_ago].copy()

    return df, df_recent

def make_report(dataframe1,dataframe2):
    report_pivot01 = pd.pivot_table(dataframe1,
                                index=['年度','月度'],
                                columns='予約内容',
                                values='稼働時間_hour',
                                aggfunc='sum',
                                fill_value=0)
    report_pivot02 = pd.pivot_table(dataframe2,
                                 index='名前_統一',
                                 columns='予約内容',
                                 values='稼働時間_hour',
                                 aggfunc='sum',
                                 fill_value=0)
    report_pivot03 = pd.pivot_table(dataframe2,
                                    index = '名前_統一',
                                    columns =['年度','月度'],
                                    values = '稼働時間_hour',
                                    aggfunc='sum',
                                    fill_value=0)

    return report_pivot01,report_pivot02,report_pivot03

def visualize_report(report01,report02,report03):
    plt.figure(figsize=(12,6))
    report01.plot(kind='line', ax=plt.gca(),marker='o')
    plt.title('月度別・社有車別 稼働時間の推移')
    plt.xlabel('(年度,月度)')
    plt.ylabel('総稼働時間(時間)')
    plt.legend()
    plt.grid(True,linestyle='--',alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'company_car_usage_{today}.png')
    plt.show

    plt.figure(figsize=(12,10))
    sns.heatmap(
        report02,
        annot=True,
        fmt='.1f',
        cmap='Blues',
        linewidths=.5,
        cbar_kws={'label':'総稼働時間(Hour)'}
    )
    plt.title('利用者別・社有車別 稼働時間ヒートマップ', fontsize=16)
    plt.xlabel('車両ナンバー',fontsize=12)
    plt.ylabel('社員名', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'company_car_heatmap_{today}.png')
    plt.show

    total_usage_by_user = report03.sum(axis=1)
    sorted_index = total_usage_by_user.sort_values(ascending=True).index
    report03_sorted = report03.reindex(sorted_index)

    plt.figure(figsize=(20,10))
    report03_sorted.plot.barh()
    plt.title('月度別・利用者別 総稼働時間(Hours)')
    plt.xlabel('総稼働時間(時間)')
    plt.xticks(rotation=45)
    plt.ylabel('利用者')
    plt.legend(title='月度',bbox_to_anchor=(1.05,1),loc='upper left')
    plt.grid(axis='y',linestyle='--',alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'company_car_user_{today}.png')
    plt.show()


def export_report_to_excel(pivot_table,file_name,sheet_name):
    car_list = [
        '和泉581み9657 (積算部管理)',
        '和泉581は5240 (積算部管理)',
        '和泉581む1869 (積算部管理)',
        '和泉581く9368 (安品ア室管理)',
        '和泉581の6302 (安品ア室管理)'
        ]
    pivot_table = pivot_table[car_list]
    with pd.ExcelWriter(file_name, engine='openpyxl', mode='w') as writer:
        pivot_table.to_excel(writer,sheet_name = sheet_name)



df = load_and_combine_reservations()
(repo1,repo2,repo3) = make_report(df)
visualize_report(repo1,repo2,repo3)
export_report_to_excel(repo1,export_file,'月別_社有車別_稼働時間')



KeyError: '利用終開始日時'